In [1]:
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

import time
import json
import ctypes
import struct
import blosc2
import numpy as np
import numpy as np
from pathlib import Path
import sys

In [2]:
sys.path.append('/projects/insituperf/seer_o/venv-seer/lib/python3.11/site-packages')

In [3]:
sys.path.append('/projects/insituperf/SZ3/tools/pysz/')
from pysz import SZ

In [4]:
import pyvista as pv

In [5]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [6]:
def list_keys(db, simid):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            x.append(k)

    return x

In [7]:
def list_fields(db, simid):
    
    all_keys = list_all_keys(db)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            if len(parts) == 5:
                name = parts[len(parts)-2]
                x.append(name)
        
    return list(set(x))

In [8]:
def list_attributes(db, simid):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            if len(parts) == 5:
                name = parts[len(parts)-1]
                x.append(name)

    return list(set(x))

In [9]:
def put_value(db, key, value):
    ''' Put data into the server for that key '''
    db.put(key=key, value=value)  # get the data


In [24]:
def get_raw_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)
    #print(l)
    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    #print(type(out_val))
    return out_val

In [11]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [12]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [13]:
def get_decompDataBLOSC(db, key, num_elems):
        x = []
        
        val = get_data(db, key)
        a_bytesobj2 = blosc2.decompress(val)
        
        bf = str(num_elems) + 'f'
        x = struct.unpack(bf, a_bytesobj2)

        return x

In [14]:
def get_decompDataSZ3(db, key, num_elems):
        data = get_data(db, key)
        np_array = np.frombuffer(data, dtype=np.uint8)
        
        lib_extention = {
            "darwin": "libSZ3c.dylib",
            "windows": "SZ3c.dll",
        }.get(sys.platform, "libSZ3c.so")

        sz = SZ("/projects/insituperf/SZ3/install/lib64/{}".format(lib_extention))
        
        data_dec = sz.decompress(np_array, (num_elems,1,1), np.float32)


        return data_dec

In [15]:
def isTsReady(db, ts, simid):
    key = '_' + simid + '/' + ts +'/status'
    return get_value(db, key)

In [16]:
def getNumRanks(db, simid):
    key = '_' + simid + '/num_ranks'
    return get_value(db, key)

In [17]:
f = open('/projects/insituperf/seer_o/test_app/mochi-yokan-config.json')
#f = open('/vast/projects/exasky/pascal/HACC/trunk/mochi-yokan-config.json')

In [18]:
json_data = json.load(f)
json_data

{'sim-id': '08_000',
 'libraries': {'yokan': '/vast/home/pascalgrosset/spack/opt/spack/linux-rhel8-haswell/gcc-9.4.0/mochi-yokan-0.4.2-hiu7yh7om6nmyc2ahuknpdsov5k64zcj/lib/libyokan-bedrock-module.so'},
 'providers': [{'name': 'yokan_provider',
   'provider_id': 124,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map'}}}],
 'data': [{'name': 'pressure_3', 'compressor': 'BLOSC'},
  {'name': 'temperature_3', 'compressor': 'SZ3', 'mode': 'psnr', 'value': 50},
  {'name': 'energy_3', 'compressor': 'SZ3', 'mode': 'psnr', 'value': 75}],
 'databases': [{'address': '192.168.81.82:41009',
   'protocol': 'ofi+tcp',
   'provider_id': 124}]}

In [19]:
server_addr1 = "ofi+tcp://192.168.81.82:41009"
provider_id = 124
protocol = 'ofi+tcp'

In [20]:
engine1 = Engine(protocol)
mid1 = engine1.get_internal_mid()
addr1 = engine1.lookup(server_addr1)
hg_addr1 = addr1.get_internal_hg_addr()
provider1 = Provider(mid=mid1, provider_id=provider_id, config='{"database":{"type":"map"}}')
client1 = Client(mid=mid1)
db1 = client1.make_database_handle(address=hg_addr1, provider_id=provider_id)

In [ ]:
server_addr2 = "ofi+tcp://192.168.81.78:45745"
provider_id = 124
protocol = 'ofi+tcp'

In [ ]:
engine2 = Engine(protocol)
mid2 = engine2.get_internal_mid()
addr2 = engine2.lookup(server_addr2)
hg_addr2 = addr2.get_internal_hg_addr()
provider2 = Provider(mid=mid2, provider_id=provider_id, config='{"database":{"type":"map"}}')
client2 = Client(mid=mid2)
db2 = client2.make_database_handle(address=hg_addr2, provider_id=provider_id)

In [ ]:
dbs = []

In [ ]:
dbs.append(db1)


In [ ]:
dbs.append(db2)

In [ ]:
arr = np.random.rand(10)
print(arr)
print(type(arr[0]))

a_bytes = arr.tobytes()
print(type(a_bytes))

aa_bytes = bytearray(arr)
print(type(aa_bytes))


print(len(aa_bytes))
put_value(db1, "1", aa_bytes)

y = np.frombuffer(aa_bytes, dtype=np.float64)
print(y)


# print(len(a_bytes))
# put_value(db1, "1", a_bytes)

# y = np.frombuffer(a_bytes, dtype=np.float64)
# print(y)

In [ ]:
x = get_raw_value(db1, "1")
y = np.frombuffer(x, dtype=np.float64)
y

In [ ]:
print("num_elements, size, time")
for i in range(50):
    num_elements = 2000000*i
    arr = np.random.rand(num_elements)
    aa_bytes = bytearray(arr)
    print(num_elements, len(aa_bytes))

In [21]:
from sys import getsizeof

print("num_elements, size, time")
for i in range(50):
    num_elements = 2000000*i
    arr = np.random.rand(num_elements)
    aa_bytes = bytearray(arr)

    start = time.time()
    put_value(db1, str(i), aa_bytes)
    end = time.time()
    #print("time" , end - start, "num_elements:", num_elements, "size:", (getsizeof(arr)))
    print(num_elements, ",", len(aa_bytes), ",", (end - start))



num_elements, size, time
0 , 0 , 0.0004525184631347656
2000000 , 16000000 , 0.026096582412719727
4000000 , 32000000 , 0.05001473426818848
6000000 , 48000000 , 0.07336854934692383
8000000 , 64000000 , 0.09303832054138184
10000000 , 80000000 , 0.1046135425567627
12000000 , 96000000 , 0.1271531581878662
14000000 , 112000000 , 0.1456282138824463
16000000 , 128000000 , 0.16367149353027344
18000000 , 144000000 , 0.18999743461608887
20000000 , 160000000 , 0.19549250602722168
22000000 , 176000000 , 0.20886754989624023
24000000 , 192000000 , 0.25335121154785156
26000000 , 208000000 , 0.26697349548339844
28000000 , 224000000 , 0.2585794925689697
30000000 , 240000000 , 0.2996804714202881
32000000 , 256000000 , 0.31120753288269043
34000000 , 272000000 , 0.3140091896057129
36000000 , 288000000 , 0.36887121200561523
38000000 , 304000000 , 0.36498141288757324
40000000 , 320000000 , 0.38431692123413086
42000000 , 336000000 , 0.3887350559234619
44000000 , 352000000 , 0.3827688694000244
46000000 , 36800

In [ ]:
#get_value(dbs[0],"1")

In [27]:
print("# , time")
for i in range(50):
    start = time.time()
    x = get_raw_value(db1, str(i))
    end = time.time()
    print(i, ",", (end - start))

# , time
0 , 0.00531315803527832
1 , 0.015767812728881836
2 , 0.030086755752563477
3 , 0.07719755172729492
4 , 0.10656070709228516
5 , 0.13657927513122559
6 , 0.16431784629821777
7 , 0.18913483619689941
8 , 0.21128225326538086
9 , 0.2393648624420166
10 , 0.26108598709106445
11 , 0.2897801399230957
12 , 0.3089008331298828
13 , 0.31063199043273926
14 , 0.3610365390777588
15 , 0.36743617057800293
16 , 0.3867990970611572
17 , 0.4111347198486328
18 , 0.43905162811279297
19 , 0.41718506813049316
20 , 0.4920766353607178
21 , 0.44118189811706543
22 , 0.4895784854888916
23 , 0.5136599540710449
24 , 0.5215380191802979
25 , 0.5262501239776611
26 , 0.5701560974121094
27 , 0.5693590641021729
28 , 0.6004137992858887
29 , 0.6253054141998291
30 , 0.6420567035675049
31 , 0.6441998481750488
32 , 0.6573245525360107
33 , 0.671377420425415
34 , 0.7025647163391113
35 , 0.7372410297393799
36 , 0.7238314151763916
37 , 0.7547247409820557
38 , 0.761918306350708
39 , 0.8080735206604004
40 , 0.8163192272186279
41

In [ ]:
ts = '0'

In [ ]:
simid = '07730'

In [ ]:
keys = list_all_keys(dbs[0])
keys

In [ ]:
list_keys(dbs[0],'56789')

In [ ]:
fields = list_fields(dbs[0], simid)
fields

In [ ]:
attributes = list_attributes(dbs[0], simid)
attributes

In [ ]:
isTsReady(dbs[0], '1', simid)

In [ ]:
get_value(dbs[0], '_07730/0/status',)

In [ ]:
getNumRanks(dbs[0], simid)

In [ ]:
get_value(dbs[0], "_07720/0/1/pressure_3/dbIndex")

In [ ]:
get_value(dbs[0], "_07720/1/1/pressure_3/dbIndex")

In [ ]:
nE = get_value(dbs[0],"_07720/0/1/pressure_3/num_elems")
nE

In [ ]:
get_value(dbs[0],"_0777/4/1/pressure_3/num_elems")

In [ ]:
pressure_3 = get_decompDataBLOSC(dbs[0], "_0777/4/1/pressure_3/value", 100)
pressure_3

In [ ]:
pressure_3 = get_decompDataBLOSC(dbs[1], "_07720/1/1/pressure_3/value", 100)
pressure_3

In [ ]:
len(com_x)

In [ ]:
get_value(db, "_56789/499/0/x/num_elems")

In [ ]:
val_x = get_decompDataSZ3(db, "_56789/499/0/x/value", 3080753)
vals_x = val_x.flatten()

In [ ]:
val_y = get_decompDataSZ3(db, "_56789/499/0/y/value", 3080753)
vals_y = val_y.flatten()

In [ ]:
val_z = get_decompDataSZ3(db, "_56789/499/0/z/value", 3080753)
vals_z = val_z.flatten()

In [ ]:
xxx= np.stack([vals_x,vals_y,vals_z], axis=1)

In [ ]:
import pyvista as pv
from pyvista import examples
from pyvista.trame.jupyter import elegantly_launch, launch_server

pv.global_theme.trame.server_proxy_enabled = True
pv.global_theme.trame.server_proxy_prefix = 'darwin-fe.lanl.gov:8879' # white screen

points = xxx
point_cloud = pv.PolyData(points)
point_cloud.plot(jupyter_backend='trame', eye_dome_lighting=True)
point_cloud.point_size = 0.01
point_cloud.opacity = 0.1

plotter = pv.Plotter(notebook=True)
plotter.add_mesh(point_cloud,opacity=0.25,point_size=1.0,color='#0000ff')
plotter.camera.zoom(4.0)
plotter.show(jupyter_backend='static') #works screen

In [ ]:
np.savetxt("/projects/insituperf/seer_o/3d_array.csv", xxx, delimiter=",")

In [ ]:
np.save('/projects/insituperf/seer_o/my_array.npy', xxx)

In [ ]:
import pyvista as pv

In [ ]:
%pip list

In [ ]:
!{sys.executable} -m pip install 'pyvista[jupyter]>=0.38.1'